# 手撕 Unigram Language Model 分词

## 背景
Unigram 假设每个 token 独立生成，用 EM 训练词表概率。
编码时用 Viterbi 找最大概率分词路径。
SentencePiece 默认使用 Unigram 模型。

## 考察点
- Unigram 概率模型：P(word) = prod P(token_i)
- Viterbi 分词（动态规划找最优切分）
- EM 训练（简化版）

In [ ]:
import math
from collections import Counter

class UnigramTokenizer:
    def __init__(self, vocab: int = None) -> None:
        # vocab: {token: log_prob}
        self.vocab = vocab or {}

    def train(self, texts, vocab_size: int = 100) -> None:
        # 简化训练：枚举所有子串，按频率估计概率
        substr_freqs = Counter()
        for text in texts:
            for word in text.split():
                for i in range(len(word)):
                    for j in range(i + 1, len(word) + 1):
                        substr_freqs[word[i:j]] += 1
        # 取 top-k 子串作为词表
        top = substr_freqs.most_common(vocab_size)
        total = sum(f for _, f in top)
        self.vocab = {tok: math.log(f / total) for tok, f in top}

    def encode_word(self, word: str) -> torch.Tensor:
        # Viterbi: dp[i] = 最佳分词 word[:i] 的 log_prob
        n = len(word)
        dp = [-float('inf')] * (n + 1)
        dp[0] = 0.0
        back = [0] * (n + 1)
        for end in range(1, n + 1):
            for start in range(end):
                sub = word[start:end]
                if sub in self.vocab and dp[start] + self.vocab[sub] > dp[end]:
                    dp[end] = dp[start] + self.vocab[sub]
                    back[end] = start
        # 回溯
        tokens = []
        pos = n
        while pos > 0:
            tokens.append(word[back[pos]:pos])
            pos = back[pos]
        return tokens[::-1]

    def encode(self, text: str) -> torch.Tensor:
        tokens = []
        for word in text.split():
            tokens.extend(self.encode_word(word))
        return tokens

In [ ]:
# 验证 Unigram
texts = ["hello world", "hello hello", "world world world"]
ut = UnigramTokenizer()
ut.train(texts, vocab_size=50)
encoded = ut.encode("hello world")
print(f"encode('hello world'): {encoded}")
assert len(encoded) > 0
# 验证 Viterbi 找到合理切分
assert "".join(encoded) == "helloworld", "切分拼接应还原原文"
print("✅ Unigram Viterbi 分词验证通过")